# Testing methods to compare between two models

## Playing around with gene weights dataframes to compare gene weights between models.

In [1]:
import pandas as pd

In [2]:
# read in the csv files of the threshold rna-seq data
stm_threshold_10_90 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_25_75.csv")
mixed_stm_threshold_10_90 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/mixed_stm_threshold_10_90.csv")
mixed_stm_threshold_15_85 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/mixed_stm_threshold_15_85.csv")
mixed_stm_threshold_25_75 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/mixed_stm_threshold_25_75.csv")

In [3]:
stm_threshold_10_90

,genes,threshold_category
0,STM0001,0
1,STM0002,0
2,STM0003,0
3,STM0004,0
4,STM0005,0
...,...,...
4666,PSLT106,-1
4667,PSLT107,-1
4668,PSLT108,-1
4669,PSLT110,-1


In [4]:
df = stm_threshold_10_90.merge(stm_threshold_15_85, on = "genes", how = "left")
df

,genes,threshold_category_x,threshold_category_y
0,STM0001,0,0
1,STM0002,0,0
2,STM0003,0,0
3,STM0004,0,0
4,STM0005,0,0
...,...,...,...
4666,PSLT106,-1,-1
4667,PSLT107,-1,-1
4668,PSLT108,-1,-1
4669,PSLT110,-1,-1


In [5]:
df[df.threshold_category_x != df.threshold_category_y]

,genes,threshold_category_x,threshold_category_y
13,STM0014,0,-1
21,STM0022,0,-1
24,STM0025,0,-1
26,STM0027,0,-1
27,STM0028,0,-1
...,...,...,...
4530,STM4562,0,1
4541,STM4573,0,-1
4549,STM4581,0,1
4554,STM4586,0,1


## Playing around with reaction weights

In [6]:
import os
import sys
import argparse
import numpy as np
import pandas as pd

# COBRApy
from cobra.core.configuration import Configuration
from cobra.io import load_model
from cobra.io import save_json_model

#iMATpy
from imatpy.imat import imat
from imatpy.model_creation import generate_model
from imatpy.parse_gpr import gene_to_rxn_weights

In [7]:
os.environ["GRB_LICENSE_FILE"] = "/home/gmvaz/.gurobi.lic"
Configuration().solver = "gurobi"

In [8]:
# load the test S.Tm LT2 GEM in cobrapy
base_model = load_model("STM_v1_0")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2789960
Academic license 2789960 - for non-commercial use only - registered to gm___@ucdavis.edu


In [9]:
# read in the csv files of the threshold rna-seq data
stm_threshold_10_90 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_10_90.csv")
stm_threshold_15_85 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_15_85.csv")
stm_threshold_25_75 = pd.read_csv("/home/gmvaz/2026_GEMs/imat_prep_test/stm_threshold_25_75.csv")

In [10]:
# make each list into a python set
stm_set_10_90 = set(stm_threshold_10_90["genes"].astype(str).str.strip())
stm_set_15_85 = set(stm_threshold_15_85["genes"].astype(str).str.strip())
stm_set_25_75 = set(stm_threshold_25_75["genes"].astype(str).str.strip())

#check
print(len(stm_set_10_90))
print(len(stm_set_15_85))
print(len(stm_set_25_75))

4671
4671
4671


In [11]:
# make a list of all genes in the input model and save it as a set
model_genes = set([g.id for g in base_model.genes])

#check
print(len(model_genes))

1271


In [12]:
# find the genes that exist in the input model and gene weights file
overlap_10_90 = stm_set_10_90.intersection(model_genes)
overlap_15_85 = stm_set_15_85.intersection(model_genes)
overlap_25_75 = stm_set_25_75.intersection(model_genes)

#check
print(len(overlap_10_90))
print(len(overlap_15_85))
print(len(overlap_25_75))

1251
1251
1251


In [13]:
# find genes that exist in the input model but do not appear in the gene weights file
only_model_stm1090 = model_genes.difference(stm_set_10_90)
only_model_stm1585 = model_genes.difference(stm_set_15_85)
only_model_stm2575 = model_genes.difference(stm_set_25_75)

# check
print(len(only_model_stm1090))
print(len(only_model_stm1585))
print(len(only_model_stm2575))

20
20
20


In [14]:
# find genes that exist in the gene weights file but are not in the input model
only_stm_stm1090 = stm_set_10_90.difference(model_genes)
only_stm_stm1585 = stm_set_15_85.difference(model_genes)
only_stm_stm2575 = stm_set_25_75.difference(model_genes)

# check
print(len(only_stm_stm1090))
print(len(only_stm_stm1585))
print(len(only_stm_stm2575))

3420
3420
3420


In [15]:
# subset the gene weights list for each model/sample to only have the genes present in the input model
stm1090_model = stm_threshold_10_90[stm_threshold_10_90["genes"].isin(model_genes)]
stm1585_model = stm_threshold_15_85[stm_threshold_15_85["genes"].isin(model_genes)]
stm2575_model = stm_threshold_25_75[stm_threshold_25_75["genes"].isin(model_genes)]

In [16]:
# there are different amounts of genes in the gene weights lists and in the input model gene list. 
# make a new gene weight list per sample that has all the genes from the input model:
# (1) assign a value of 0 to the 20 genes found in the input model but not in the gene weights list. 
# (2) add the genes that are present in the input model and raw gene weight list. 
# Result: a gene weight list per sample that has only the genes found in the input model and raw gene weights list. 
subset_weights_stm1090 = stm1090_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm1585 = stm1585_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)
subset_weights_stm2575 = stm2575_model.set_index("genes")["threshold_category"].reindex(model_genes, fill_value=0)

# check
print(len(subset_weights_stm1090))
print(len(subset_weights_stm1585))
print(len(subset_weights_stm2575))

1271
1271
1271


In [17]:
# convert gene weights to reaction weights using the final gene weights lists with all genes from the input model
rxn_weights_stm1090 = gene_to_rxn_weights(base_model, subset_weights_stm1090)
rxn_weights_stm1585 = gene_to_rxn_weights(base_model, subset_weights_stm1585)
rxn_weights_stm2575 = gene_to_rxn_weights(base_model, subset_weights_stm2575)

# check
print(len(rxn_weights_stm1090))
print(len(rxn_weights_stm1585))
print(len(rxn_weights_stm2575))

2545
2545
2545


In [18]:
stm1090_model = generate_model(model = base_model, rxn_weights = rxn_weights_stm1090, method = "imat_restrictions")
stm1585_model = generate_model(model = base_model, rxn_weights = rxn_weights_stm1585, method = "imat_restrictions")
stm2575_model = generate_model(model = base_model, rxn_weights = rxn_weights_stm2575, method = "imat_restrictions")

Read LP format model from file /tmp/tmpxvtkk12h.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpbg6vedeg.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros
Read LP format model from file /tmp/tmpedja8ky9.lp
Reading time = 0.02 seconds
: 1802 rows, 5090 columns, 19612 nonzeros


In [19]:
save_json_model(stm1090_model, "/home/gmvaz/2026_GEMs/stm_model_test/run_051326/stm1090_imat_restrictions.json")
save_json_model(stm1585_model, "/home/gmvaz/2026_GEMs/stm_model_test/run_051326/stm1585_imat_restrictions.json")
save_json_model(stm2575_model, "/home/gmvaz/2026_GEMs/stm_model_test/run_051326/stm2575_imat_restrictions.json")

In [20]:
print(type(rxn_weights_stm1090))
rxn_weights_stm1090

<class 'pandas.core.series.Series'>


3HAD180      0.0
3HAD181      0.0
3HAD40       0.0
3HAD60       0.0
3HAD80       0.0
            ... 
12dgr2_ST    0.0
OAL_ST       0.0
OA4L_ST      0.0
OA5L_ST      0.0
OA4VL_ST     0.0
Length: 2545, dtype: float64

In [21]:
df_stm1090 = rxn_weights_stm1090.to_frame()
df_stm1585 = rxn_weights_stm1585.to_frame()
df_stm2575 = rxn_weights_stm2575.to_frame()

In [22]:
df_stm1090

,0
3HAD180,0.0
3HAD181,0.0
3HAD40,0.0
3HAD60,0.0
3HAD80,0.0
...,...
12dgr2_ST,0.0
OAL_ST,0.0
OA4L_ST,0.0
OA5L_ST,0.0


In [23]:
# merges on index column by default
df = pd.concat([df_stm1090, df_stm1585], axis = 1)
df.columns = ["a", "b"]

In [24]:
df

,a,b
3HAD180,0.0,0.0
3HAD181,0.0,0.0
3HAD40,0.0,0.0
3HAD60,0.0,0.0
3HAD80,0.0,0.0
...,...,...
12dgr2_ST,0.0,0.0
OAL_ST,0.0,0.0
OA4L_ST,0.0,0.0
OA5L_ST,0.0,0.0


In [25]:
stm1090_vs1585_rxnweights = df[df.a != df.b]
stm1090_vs1585_rxnweights

,a,b
ACNAMt2pp,0.0,1.0
ALLTN,0.0,1.0
AOXSr,0.0,-1.0
ARAI,0.0,1.0
ARBt2rpp,0.0,1.0
...,...,...
OCDCEAt2pp,0.0,1.0
MUREINLPPTP,0.0,1.0
2HH24DDH1,0.0,-1.0
GALCTND,0.0,1.0


In [26]:
stm1090_vs1585_rxnweights.to_csv('stm1090_vs1585_rxnweights.csv', index=True)

# check reaction bounds

- check reaction bounds in input model
-  does imat_restrictions method change bounds?

In [27]:
for rx in stm1090_model.reactions:
    print(rx.lower_bound, rx.upper_bound, rx.id, rx.name)
    break

0.0 1000.0 3HAD180 3-hydroxyacyl-[acyl-carrier-protein] dehydratase (n-C18:0)


In [28]:
stm1090_model.reactions

[<Reaction 3HAD180 at 0x132eb309eb50>,
 <Reaction 3HAD181 at 0x132eb2d13d10>,
 <Reaction 3HAD40 at 0x132eb309d190>,
 <Reaction 3HAD60 at 0x132eb3094e90>,
 <Reaction 3HAD80 at 0x132eb3093490>,
 <Reaction 3KGK at 0x132eb2cf3210>,
 <Reaction 3NTD2pp at 0x132eb3093a10>,
 <Reaction 3NTD4pp at 0x132eb3092d10>,
 <Reaction 3NTD7pp at 0x132eb3092050>,
 <Reaction 3NTD9pp at 0x132eb3093bd0>,
 <Reaction 3OAR100 at 0x132eb308b950>,
 <Reaction 3OAR120 at 0x132eb308bf90>,
 <Reaction 3OAR121 at 0x132eb3089a10>,
 <Reaction 3OAR140 at 0x132eb3089f10>,
 <Reaction 3OAR141 at 0x132eb3089910>,
 <Reaction 3OAR160 at 0x132eb30892d0>,
 <Reaction 3OAR161 at 0x132eb3089090>,
 <Reaction 23CAMPtex at 0x132eb30904d0>,
 <Reaction 23CCMPtex at 0x132eb3080b50>,
 <Reaction 23CGMPtex at 0x132eb3080310>,
 <Reaction 12DGR120tipp at 0x132eb3077510>,
 <Reaction 12DGR140tipp at 0x132eb3077550>,
 <Reaction 12DGR141tipp at 0x132eb3076b90>,
 <Reaction 12DGR160tipp at 0x132eb3076d90>,
 <Reaction 23CUMPtex at 0x132eb3077150>,
 <R

In [29]:
# create two dictionaries in python. dictionaries store key-value pairs
lower_bounds = {}
upper_bounds = {}

for rx in stm1090_model.reactions:
    lower_bounds[rx.id] = rx.lower_bound
    upper_bounds[rx.id] = rx.upper_bound

In [30]:
lower_bounds

{'3HAD180': 0.0,
 '3HAD181': 0.0,
 '3HAD40': 0.0,
 '3HAD60': 0.0,
 '3HAD80': 0.0,
 '3KGK': 0.0,
 '3NTD2pp': 0.0,
 '3NTD4pp': 0.0,
 '3NTD7pp': 0.0,
 '3NTD9pp': 0.0,
 '3OAR100': -1000.0,
 '3OAR120': -1000.0,
 '3OAR121': 0.0,
 '3OAR140': -1000.0,
 '3OAR141': 0.0,
 '3OAR160': -1000.0,
 '3OAR161': 0.0,
 '23CAMPtex': -1000.0,
 '23CCMPtex': -1000.0,
 '23CGMPtex': -1000.0,
 '12DGR120tipp': 0.0,
 '12DGR140tipp': 0.0,
 '12DGR141tipp': 0.0,
 '12DGR160tipp': 0.0,
 '23CUMPtex': -1000.0,
 '3OAR180': -1000.0,
 '23DAPPAt2pp': 0.0,
 '23DAPPAtex': -1000.0,
 '12DGR161tipp': 0.0,
 '12DGR180tipp': 0.0,
 '12DGR181tipp': 0.0,
 '12PPDRtex': -1000.0,
 '12PPDRtpp': -1000.0,
 '12PPDStex': -1000.0,
 '12PPDStpp': -1000.0,
 '23PDE2pp': 0.0,
 '23PDE4pp': 0.0,
 '23PDE7pp': 0.0,
 '23PDE9pp': 0.0,
 '26DAHtex': -1000.0,
 '2AGPA120tipp': 0.0,
 '2AGPA140tipp': 0.0,
 '2AGPA141tipp': 0.0,
 '2AGPA160tipp': 0.0,
 '2AGPA161tipp': 0.0,
 '2AGPA180tipp': 0.0,
 '2AGPA181tipp': 0.0,
 '14GLUCANabcpp': 0.0,
 '14GLUCANtexi': 0.0,
 '2A

In [31]:
for rx2 in stm1585_model.reactions:
    rx1_lower = lower_bounds[rx2.id]
    rx1_upper = upper_bounds[rx2.id]
    if 1 or rx2.lower_bound != rx1_lower or rx2.upper_bound != rx1_upper:
        print(rx2.id, rx2.lower_bound, rx1_lower, rx2.upper_bound, rx1_upper)

3HAD180 0.0 0.0 1000.0 1000.0
3HAD181 0.0 0.0 1000.0 1000.0
3HAD40 0.0 0.0 1000.0 1000.0
3HAD60 0.0 0.0 1000.0 1000.0
3HAD80 0.0 0.0 1000.0 1000.0
3KGK 0.0 0.0 1000.0 1000.0
3NTD2pp 0.0 0.0 1000.0 1000.0
3NTD4pp 0.0 0.0 1000.0 1000.0
3NTD7pp 0.0 0.0 1000.0 1000.0
3NTD9pp 0.0 0.0 1000.0 1000.0
3OAR100 -1000.0 -1000.0 1000.0 1000.0
3OAR120 -1000.0 -1000.0 1000.0 1000.0
3OAR121 0.0 0.0 1000.0 1000.0
3OAR140 -1000.0 -1000.0 1000.0 1000.0
3OAR141 0.0 0.0 1000.0 1000.0
3OAR160 -1000.0 -1000.0 1000.0 1000.0
3OAR161 0.0 0.0 1000.0 1000.0
23CAMPtex -1000.0 -1000.0 1000.0 1000.0
23CCMPtex -1000.0 -1000.0 1000.0 1000.0
23CGMPtex -1000.0 -1000.0 1000.0 1000.0
12DGR120tipp 0.0 0.0 1000.0 1000.0
12DGR140tipp 0.0 0.0 1000.0 1000.0
12DGR141tipp 0.0 0.0 1000.0 1000.0
12DGR160tipp 0.0 0.0 1000.0 1000.0
23CUMPtex -1000.0 -1000.0 1000.0 1000.0
3OAR180 -1000.0 -1000.0 1000.0 1000.0
23DAPPAt2pp 0.0 0.0 1000.0 1000.0
23DAPPAtex -1000.0 -1000.0 1000.0 1000.0
12DGR161tipp 0.0 0.0 1000.0 1000.0
12DGR180tipp 0.0

# check simulation results

## FBA Overall Growth Rate for all models

In [32]:
fba_stm1090 = stm1090_model.optimize()
fba_stm1585 = stm1585_model.optimize()
fba_stm2575 = stm2575_model.optimize()
fba_base = base_model.optimize()

In [33]:
stm1090_model.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ca2_e,EX_ca2_e,0.001929,0,0.00%
cl_e,EX_cl_e,0.001929,0,0.00%
cobalt2_e,EX_cobalt2_e,0.001286,0,0.00%
cu2_e,EX_cu2_e,0.001286,0,0.00%
fe2_e,EX_fe2_e,0.01193,0,0.00%
fe3_e,EX_fe3_e,0.0009595,0,0.00%
glc__D_e,EX_glc__D_e,5,6,100.00%
k_e,EX_k_e,0.07232,0,0.00%
mg2_e,EX_mg2_e,0.003215,0,0.00%
mn2_e,EX_mn2_e,0.001286,0,0.00%


In [34]:
print(fba_base.objective_value)
print(fba_stm1090.objective_value)
print(fba_stm1585.objective_value)
print(fba_stm2575.objective_value)

0.4778336607607207
0.40720580512198207
0.40650099210576757
0.39342197276970997


# Making sense of the predicted growth rates

What is growth rate in FBA? 

What does the number mean biologically? 

What growth rate values should we expect? 

How can we compare between values?
Let's use the predicted growth rate of the input model as a control to gauge how different the predictions are for the extracted models. I'll divide the growth rate of the input model by the growth rate of each extracted model to see the percent change.


# Calculating percent decrease and percent retained

In [35]:
# (1) calculate the difference in growth prediction. base model - extracted model = difference
# (2) divide the difference by the base model
# (3) convert the result of each division to a percentage (multiply by 100) 

# Coding work
Inspect the variables I'm using first. 
```
type(fba_base.objective_value) = float
dir(fba_base.objective_value)
print(fba_base.objective_value) = 0.4778336607607207
```
From this I know that my objective values are float (numbers with a decimal point) and I know that when I print it out, I get a long decimal. 

I want to (1) have a math equation that solves the difference of extracted model and the base model and (2) takes the difference and calculates what percentage it is of the base model. The final answer should be to the second decimal point. 


In [36]:
# calculate percent difference for each extracted model 
# percent_decrease = ((input_model - extracted_model) / input_model) * 100

stm1090_decrease = ((fba_base.objective_value - fba_stm1090.objective_value) / (fba_base.objective_value))
print(stm1090_decrease)

# The value assigned to the variable `stm1090_decrease` is a float. 
# I need to multiply by 100, convert to a percent, and round to the second decimal point. 

#  `format()` is a built in python function that converts the value to a string-representation of a specified format. 
# in this case, I am turning a decimal number into a percentage string with two digits after the decimal
# `format(float, ".2%") - `float` is the number that I want to change. "%" is saying take this decimal number, multiply it by 100,
# and add a percent sign. The `.2` is saying round to the second decimal point. 

stm1090_percentdecrease = format(stm1090_decrease, ".2%")
print(stm1090_percentdecrease)

stm1585_decrease = ((fba_base.objective_value - fba_stm1585.objective_value) / (fba_base.objective_value))
print(stm1585_decrease)
stm1585_percentdecrease = format(stm1585_decrease, ".2%")
print(stm1585_percentdecrease)

stm2575_decrease = ((fba_base.objective_value - fba_stm2575.objective_value) / (fba_base.objective_value))
print(stm2575_decrease)
stm2575_percentdecrease = format(stm2575_decrease, ".2%")
print(stm2575_percentdecrease)

0.1478084560353025
14.78%
0.14928347354472707
14.93%
0.17665496368888212
17.67%


In [37]:
# I can also calculate the percent of growth retained from the input model
# percent retained = (extracted_model / input_model) * 100

stm1090_percentretained = fba_stm1090.objective_value / fba_base.objective_value
print(format(stm1090_percentretained, ".2%"))

stm1585_percentretained = fba_stm1585.objective_value / fba_base.objective_value
print(format(stm1585_percentretained, ".2%"))

stm2575_percentretained = fba_stm2575.objective_value / fba_base.objective_value
print(format(stm2575_percentretained, ".2%"))

85.22%
85.07%
82.33%


Models `stm1090_model` and `stm1585_model` have predicted growth rates that are almost the same. They have a predicted growth rate of 14.78% and 14.93% lower than the input model's growth rate, respecitively. This suggests that threshold categories of 10%/90% and 15%/85% do not make a noticeable impact on overall growth rate predictions given these constraints. It is possible that the predicted growth rate could noticeably change once the environmental conditions are altered.  

In [38]:
print(f"{stm1090_percent_decrease:.2f}%")

NameError: name 'stm1090_percent_decrease' is not defined